# Lab Day 1 — Notebook (PSEUDO-CODE)

> **Đây là khung pseudo-code.** Mọi ô có `# TODO` là phần bạn phải tự viết. Hãy giữ cấu trúc mục (Part 0–4) và
> viết dự đoán/nhận xét bằng chữ của bạn. Đọc `README.md`, `GUIDE.md`, `RUBRIC.md` trước.
>
> **Quy trình làm bài**
> 1. `git pull` repo, tạo thư mục nộp: copy cả thư mục `code/` vào `submission_<MSSV>/code/`.
> 2. Mở `submission_<MSSV>/code/lab.ipynb` và làm việc **tại đó** (nên `REPO_ROOT = "../.."` đúng và ảnh/kết quả ghi vào `submission_<MSSV>/figures`, `results`).
> 3. Ở Colab: chạy ô clone repo bên dưới, rồi làm tương tự (đặt `REPO_ROOT` cho đúng).
> 4. Cuối cùng: `Restart & Run All` phải chạy hết không lỗi và giữ nguyên output.

In [ ]:
# ===== Cấu hình đường dẫn và môi trường =====
import os, sys, json, time, subprocess
import numpy as np, torch

REPO_ROOT = "../.."     # thư mục gốc repo (chứa data/ và scripts/), tính từ submission_<MSSV>/code/
OUT_DIR   = ".."        # nơi ghi figures/, results/, experiments.xlsx, predictions_eval.csv (= submission_<MSSV>/)
# Colab: nếu chưa có repo, bỏ comment 2 dòng dưới, rồi đặt REPO_ROOT = "/content/K4-Track4-Day1-Neuralnetwork"
!git clone https://github.com/VinUni-AI20k/K4-Track4-Day1-Neuralnetwork.git
REPO_ROOT = "/content/K4-Track4-Day1-Neuralnetwork"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")
if device == "cuda":
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
os.makedirs(f"{OUT_DIR}/figures", exist_ok=True)
os.makedirs(f"{OUT_DIR}/results", exist_ok=True)
from data import prepare_data
from model import MLP, EXPECTED_PARAMS, count_params, init_weights, activation_stats
from optimizer import build_optimizer, clip_gradients
from train import DEFAULT_CFG, set_seed, evaluate, predict, run_experiment, final_eval
from plots import plot_run, plot_compare
from results_table import save_result, load_results, to_row, write_xlsx

## Part 0 — Dữ liệu
Chia sẵn bằng metadata: chạy `scripts/split_data.py` từ thư mục gốc repo (tạo `data/processed/train.npz`, `eval.npz`).
Sau đó tách validation từ train, chuẩn hoá bằng thống kê của train, đưa lên device.

In [ ]:
## Part 0 — Dữ liệu

# 1. Create data/processed/train.npz and eval.npz
subprocess.run(
    [sys.executable, "scripts/split_data.py"],
    cwd=REPO_ROOT,
    check=True,
)

# 2. Load, split, standardize, and move data to device
data = prepare_data(
    device,
    val_fraction=0.2,
    seed=42,
    processed_dir=f"{REPO_ROOT}/data/processed",
)

# 3. Get the prepared tensors
X_tr = data["X_tr"]
y_tr = data["y_tr"]

X_val = data["X_val"]
y_val = data["y_val"]

X_eval = data["X_eval"]
y_eval = data["y_eval"]

# 4. Print dataset sizes
print("\nDataset sizes:")
print(f"X_tr   : {X_tr.shape}")
print(f"X_val  : {X_val.shape}")
print(f"X_eval : {X_eval.shape}")

# 5. Check standardization of the 10 numerical columns on X_tr
train_mean = X_tr[:, :10].mean(dim=0)
train_std = X_tr[:, :10].std(dim=0, unbiased=False)

print("\nX_tr numerical-feature means:")
print(train_mean)

print("\nX_tr numerical-feature standard deviations:")
print(train_std)

print("\nMean approximately 0:")
print(torch.allclose(
    train_mean,
    torch.zeros_like(train_mean),
    atol=1e-5,
))

print("Std approximately 1:")
print(torch.allclose(
    train_std,
    torch.ones_like(train_std),
    atol=1e-2,
))

## Part 1 — Model và kiểm tra "sức khoẻ" ban đầu
Quy định: `M-base` = `54 → 256 → 128 → 7`, **47 879 tham số**, logits `(B, 7)`, không softmax trong model.

In [ ]:
# ============================================================
# PART 1 — XÂY DỰNG MODEL VÀ KIỂM TRA "SỨC KHỎE" BAN ĐẦU
# ============================================================

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from model import MLP, EXPECTED_PARAMS, count_params
from train import evaluate


# ------------------------------------------------------------
# 1: Tạo M-base và kiểm tra số tham số
# ------------------------------------------------------------

model = MLP(
    hidden=(256, 128),
    dropout=0.0,
    init="he",
).to(device)

n_params = count_params(model)

print("Model:")
print(model)
print()
print("Number of parameters:", n_params)

assert n_params == EXPECTED_PARAMS[(256, 128)]

print("Parameter count check: PASS")


# ------------------------------------------------------------
# 2: Forward với một batch ngẫu nhiên (8, 54)
# ------------------------------------------------------------

x_random = torch.randn(
    8,
    54,
    device=device,
)

model.eval()

with torch.no_grad():
    logits = model(x_random)

print()
print("Input shape :", x_random.shape)
print("Logits shape:", logits.shape)

assert logits.shape == (8, 7)

print("Forward shape check: PASS")


# ------------------------------------------------------------
# 3: Step-0 loss trên validation
# ------------------------------------------------------------

step0_result = evaluate(
    model,
    data["X_val"],
    data["y_val"],
    loss_name="ce",
)

step0_loss = step0_result["loss"]
step0_acc = step0_result["acc"]
step0_f1 = step0_result["macro_f1"]

ln7 = torch.log(torch.tensor(7.0)).item()

print()
print("Step-0 validation results:")
print(f"  loss     = {step0_loss:.6f}")
print(f"  accuracy = {step0_acc:.6f}")
print(f"  macro-F1 = {step0_f1:.6f}")
print(f"  ln(7)    = {ln7:.6f}")

print()
if abs(step0_loss - ln7) < 0.1:
    print("Nhận xét: Step-0 loss gần ln(7), phù hợp với mô hình khởi tạo ban đầu.")
else:
    print("Nhận xét: Step-0 loss lệch đáng kể khỏi ln(7); cần kiểm tra initialization/model/loss.")


# ------------------------------------------------------------
# 4: Overfit 20 mẫu
# ------------------------------------------------------------

# Lấy 20 mẫu cố định từ training set
X_tiny = data["X_tr"][:20]
y_tiny = data["y_tr"][:20]

# Model mới, dropout tắt
tiny_model = MLP(
    hidden=(256, 128),
    dropout=0.0,
    init="he",
).to(device)

tiny_optimizer = torch.optim.SGD(
    tiny_model.parameters(),
    lr=0.1,
    momentum=0.9,
)

tiny_losses = []

n_steps = 500

for step in range(n_steps):

    tiny_model.train()

    logits = tiny_model(X_tiny)

    loss = F.cross_entropy(
        logits,
        y_tiny,
    )

    tiny_optimizer.zero_grad(set_to_none=True)

    loss.backward()

    tiny_optimizer.step()

    loss_value = loss.item()
    tiny_losses.append(loss_value)

    if step % 50 == 0 or step == n_steps - 1:
        print(
            f"Overfit step {step:3d}: "
            f"loss = {loss_value:.8f}"
        )

print()
print(f"Overfit initial loss: {tiny_losses[0]:.8f}")
print(f"Overfit final loss  : {tiny_losses[-1]:.8f}")


# Vẽ đường cong loss
plt.figure(figsize=(7, 4))

plt.plot(
    range(1, n_steps + 1),
    tiny_losses,
    label="training loss",
)

plt.xlabel("Step")
plt.ylabel("Cross-Entropy Loss")
plt.title("Part 1 — Overfit 20 Samples")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


# Kiểm tra loss đã giảm
assert tiny_losses[-1] < tiny_losses[0]

print()
print("Overfit check: loss decreased =", tiny_losses[-1] < tiny_losses[0])

if tiny_losses[-1] < 0.01:
    print("Overfit check: PASS — loss đã về gần 0.")
else:
    print(
        "Overfit check: loss chưa về < 0.01. "
        "Có thể tăng số bước nếu cần."
    )


# ------------------------------------------------------------
# 5: Kiểm tra gradient sau một lần backward
# ------------------------------------------------------------

grad_model = MLP(
    hidden=(256, 128),
    dropout=0.0,
    init="he",
).to(device)

# Một batch nhỏ
xb = data["X_tr"][:32]
yb = data["y_tr"][:32]

grad_model.train()

logits = grad_model(xb)

loss = F.cross_entropy(
    logits,
    yb,
)

grad_model.zero_grad(set_to_none=True)

loss.backward()

print()
print("Gradient check:")
print(f"Loss = {loss.item():.8f}")
print()

all_grad_ok = True

for name, param in grad_model.named_parameters():

    # Gradient phải tồn tại
    assert param.grad is not None, (
        f"{name}: gradient is None"
    )

    grad_norm = param.grad.norm().item()

    print(
        f"{name:30s} "
        f"grad_norm = {grad_norm:.10f}"
    )

    # Gradient phải khác 0
    if grad_norm == 0.0:
        all_grad_ok = False

    assert grad_norm != 0.0, (
        f"{name}: gradient is zero"
    )

print()
print("Gradient check: PASS")


**Nhận xét Part 1 (tự viết):** loss bước 0 đo được = 2.5154, cao hơn ln 7 = 1.946 vì ở thời điểm khởi tạo bằng He, các logit chưa phân bố gần đồng đều giữa 7 lớp nên loss không nhất thiết bằng ln(7). Quá khớp 20 mẫu: loss giảm từ 2.0363 xuống 0.0000340 sau 500 bước, cho thấy mô hình có khả năng học và vòng lặp huấn luyện hoạt động đúng.

## Part 2 — Pipeline và baseline
`run_experiment(cfg, data)` nhận một dict cấu hình và trả về lịch sử + tóm tắt. Baseline: SGD+momentum 0.9, CE, He init, batch 512, 20 epoch.

In [ ]:
# ============================================================
# BASELINE: chọn LR bằng validation + chạy nhiều seed
# ============================================================

# Các LR để thử.
# Có thể mở rộng nếu tất cả đều quá thấp/cao.
LR_CANDIDATES = [0.001, 0.003, 0.01, 0.03, 0.1]

# ------------------------------------------------------------
# 1. LR search trên validation
# ------------------------------------------------------------
lr_results = []

for lr in LR_CANDIDATES:
    cfg_lr = {
        **DEFAULT_CFG,
        "lr": lr,
        "exp_id": f"lr-search-{lr:g}",
        # Giữ seed cố định để so sánh LR công bằng.
        "seed": 1,
    }

    print("\n" + "=" * 70)
    print(f"LR SEARCH: lr={lr:g}")
    print("=" * 70)

    result = run_experiment(cfg_lr, data)
    summary = result["summary"]

    lr_results.append({
        "lr": lr,
        "val_loss": summary["best_val_loss"],
        "val_acc": summary["val_acc"],
        "val_macro_f1": summary["val_macro_f1"],
        "best_epoch": summary["best_epoch"],
        "diverged": summary["diverged"],
    })

    print(
        f"lr={lr:g} | "
        f"best_val_loss={summary['best_val_loss']:.6f} | "
        f"val_acc={summary['val_acc']:.4f} | "
        f"val_macro_f1={summary['val_macro_f1']:.4f} | "
        f"best_epoch={summary['best_epoch']} | "
        f"diverged={summary['diverged']}"
    )


# ------------------------------------------------------------
# 2. In bảng LR -> validation macro-F1
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("LR SEARCH RESULTS")
print("=" * 70)

print(
    f"{'lr':>10} "
    f"{'best_val_loss':>15} "
    f"{'val_acc':>10} "
    f"{'val_macro_f1':>15} "
    f"{'best_epoch':>12} "
    f"{'diverged':>10}"
)

for r in lr_results:
    print(
        f"{r['lr']:>10g} "
        f"{r['val_loss']:>15.6f} "
        f"{r['val_acc']:>10.4f} "
        f"{r['val_macro_f1']:>15.4f} "
        f"{r['best_epoch']:>12d} "
        f"{str(r['diverged']):>10}"
    )


# ------------------------------------------------------------
# 3. Chọn LR tốt nhất theo validation macro-F1
# ------------------------------------------------------------
valid_lr_results = [
    r for r in lr_results
    if not r["diverged"]
    and np.isfinite(r["val_macro_f1"])
]

if not valid_lr_results:
    raise RuntimeError(
        "Không có learning rate nào chạy thành công. "
        "Hãy kiểm tra run_experiment() hoặc thử LR nhỏ hơn."
    )

best_lr_result = max(
    valid_lr_results,
    key=lambda r: r["val_macro_f1"]
)

BEST_LR = best_lr_result["lr"]

print("\nSelected baseline LR:")
print(f"  lr = {BEST_LR:g}")
print(f"  val_macro_f1 = {best_lr_result['val_macro_f1']:.4f}")


# ------------------------------------------------------------
# 4. Tạo cấu hình baseline
# ------------------------------------------------------------
baseline_cfg = {
    **DEFAULT_CFG,
    "lr": BEST_LR,
}


# ------------------------------------------------------------
# 5. Chạy baseline với >= 2 seed
# ------------------------------------------------------------
BASELINE_SEEDS = [1, 2, 3]

baseline_results = []

for seed in BASELINE_SEEDS:
    exp_id = f"base-s{seed}"

    cfg = {
        **baseline_cfg,
        "seed": seed,
        "exp_id": exp_id,
    }

    print("\n" + "=" * 70)
    print(f"BASELINE: {exp_id}")
    print("=" * 70)

    result = run_experiment(cfg, data)

    baseline_results.append(result)

    summary = result["summary"]

    print(
        f"{exp_id}: "
        f"val_loss={summary['best_val_loss']:.6f}, "
        f"val_acc={summary['val_acc']:.4f}, "
        f"val_macro_f1={summary['val_macro_f1']:.4f}, "
        f"best_epoch={summary['best_epoch']}, "
        f"diverged={summary['diverged']}"
    )

    # --------------------------------------------------------
    # 6. Lưu JSON kết quả
    # --------------------------------------------------------
    save_result(
        result,
        f"{OUT_DIR}/results"
    )

    # --------------------------------------------------------
    # 7. Vẽ learning curves
    # --------------------------------------------------------
    plot_run(
        result,
        f"{OUT_DIR}/figures/{exp_id}.png"
    )


# ------------------------------------------------------------
# 8. Tính mean ± std qua các seed
# ------------------------------------------------------------
seed_macro_f1 = np.array([
    r["summary"]["val_macro_f1"]
    for r in baseline_results
], dtype=np.float64)

seed_acc = np.array([
    r["summary"]["val_acc"]
    for r in baseline_results
], dtype=np.float64)

macro_f1_mean = float(seed_macro_f1.mean())
macro_f1_std = float(seed_macro_f1.std(ddof=1)) if len(seed_macro_f1) > 1 else 0.0

acc_mean = float(seed_acc.mean())
acc_std = float(seed_acc.std(ddof=1)) if len(seed_acc) > 1 else 0.0

print("\n" + "=" * 70)
print("BASELINE SEED VARIATION")
print("=" * 70)

print(
    f"val_macro_f1 = "
    f"{macro_f1_mean:.4f} ± {macro_f1_std:.4f}"
)

print(
    f"val_acc       = "
    f"{acc_mean:.4f} ± {acc_std:.4f}"
)

print("\nApproximate 2σ noise threshold:")

print(
    f"macro-F1: ±{2 * macro_f1_std:.4f}"
)

print(
    f"accuracy : ±{2 * acc_std:.4f}"
)


# ------------------------------------------------------------
# 9. Lưu thông tin baseline để dùng tiếp ở các experiment
# ------------------------------------------------------------
baseline_stats = {
    "lr": BEST_LR,
    "seeds": BASELINE_SEEDS,
    "val_macro_f1_mean": macro_f1_mean,
    "val_macro_f1_std": macro_f1_std,
    "val_macro_f1_2sigma": 2.0 * macro_f1_std,
    "val_acc_mean": acc_mean,
    "val_acc_std": acc_std,
    "val_acc_2sigma": 2.0 * acc_std,
}

print("\nBaseline configuration:")
for k, v in baseline_cfg.items():
    print(f"  {k}: {v}")

**Nhận xét baseline (tự viết):** hình dạng đường cong (còn giảm? bắt đầu quá khớp?), val_acc so với mốc "đoán đa số", độ nhiễu giữa các seed ...

## Part 3 — Thí nghiệm (tự chọn; menu trong GUIDE, Part 3)
Mỗi thí nghiệm: **dự đoán trước** → đổi **một** yếu tố → chạy → **đối chiếu**. Lặp lại khối dưới cho từng thí nghiệm bạn chọn.
Chủ đề: loss · optimizer · hparam · dropout · clipping · amp · init.

### Thí nghiệm `<exp_id>` — chủ đề `<nhóm>`
**Dự đoán trước (viết TRƯỚC khi chạy):** ...

In [ ]:
# Mẫu một thí nghiệm. Lặp lại / bọc trong vòng for qua danh sách cfg bạn tự thiết kế.
# TODO: exp_cfg = {**cfg, "exp_id": "...", "group": "...", "description": "...", <chỉ đổi MỘT yếu tố>}
# TODO: result = run_experiment(exp_cfg, data)
# TODO: save_result(...); plot_run(...)   # mỗi thí nghiệm một ảnh figures/<exp_id>.png
# TODO: in summary (best_val_loss, val_acc, val_macro_f1, ...) và so với baseline + ngưỡng nhiễu 2σ

**Đối chiếu sau khi chạy:** khớp / khác dự đoán? vì sao (cơ chế)? chênh lệch có vượt 2σ không? ...

In [ ]:
# TODO: ảnh chồng theo nhóm: plot_compare(list_of_results_in_group, "val_loss", f"{OUT_DIR}/figures/compare_<nhóm>.png")

## Part 4 — Đánh giá cuối trên eval, bảng và nộp bài
**Cách đánh giá:** chọn cấu hình CHỈ bằng val → chạy final model → dự đoán trên **toàn bộ** tập eval → ghi `predictions_eval.csv`
→ `python scripts/evaluate.py --pred ...` tính accuracy và **macro-F1** (chỉ số chính). Chỉ làm bước này cho baseline và cấu hình cuối cùng.

In [ ]:
# TODO: chọn cấu hình cuối cùng theo VAL (có thể là baseline nếu bạn không kết hợp thêm gì)
# TODO: final_eval(cfg_final, result_final, data, f"{OUT_DIR}/predictions_eval.csv")
# TODO: chạy: python scripts/evaluate.py --pred <OUT_DIR>/predictions_eval.csv --out <OUT_DIR>/eval_result.json (từ REPO_ROOT)
# TODO: đọc eval_result.json: ghi accuracy, macro_f1 vào bảng và báo cáo; làm tương tự cho baseline (predictions riêng, không nộp nếu không cần)

In [ ]:
# TODO: phân tích lỗi trên eval: bảng precision/recall/F1 theo lớp (từ eval_result.json) và ma trận nhầm lẫn;
#       lớp nào khó nhất? vì sao (số mẫu ít? nhầm với lớp nào?)  -> viết nhận xét ở ô markdown bên dưới

**Phân tích lỗi (tự viết):** ...

In [ ]:
# TODO: results = load_results(f"{OUT_DIR}/results"); rows = [to_row(r, ...) for r in results]
# TODO: write_xlsx(rows, f"{REPO_ROOT}/templates/experiment_table_template.xlsx", f"{OUT_DIR}/experiments.xlsx")
# TODO: bổ sung sheet Seeds (exp_id các lần chạy baseline) và nhận xét ở sheet Summary